The aim of this code is to implement a basic Hi-Lo counter for blackjack. ie increment for low cards, bypass '0' cards and decrement high cards.

1. Keep all changing values together in one dictionary

In [1]:
def new_shoe(decks_total):
    """Create a fresh set of counting values for a new shoe."""
    return {
        "decks_total": decks_total,   # input from the user
        "running_count": 0,           # Hi-Lo running count starts at zero
        "cards_seen": 0,              # how many valid cards have been counted
        "processed_ids": set(),       # image names already counted 
    }

2. Reject 0, negative numbers and non numbers upfront for number of decks

In [2]:
def ask_decks():
    """Ask the user for the number of decks and keep asking until it's valid."""
    while True:                                   # repeat until we return a good value
        text = input("How many decks are in the shoe? ")   # read what the user typed
        try:
            decks = float(text)                   # allow 6 or 6.5 etc.
        except ValueError:                        # text wasn't a number at all
            print("Please enter a number, e.g. 6")
            continue                              # go back and ask again
        if decks <= 0:                            # zero or negative makes no sense
            print("Decks must be greater than 0")
            continue                              
        return decks                              

3. Convert whatever comes in to one standard form or return none if not a real rank

In [3]:
VALID_RANKS = {"A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"}

def normalise_rank(label):
    if label is None:                 # classifier gave nothing (e.g. face-down card)
        return None
    rank = str(label).strip().upper() # turn 10 -> "10", " k " -> "K"
    if rank == "1":                   # treat 1 as an ace 
        rank = "A"
    if rank in VALID_RANKS:           # only accept the 13 known ranks
        return rank
    return None                       # anything else is rejected

4. A dictionary lookup is clearer and safer than a chain of if statements for card value classification

In [4]:
HILO_VALUES = {
    "2": 1, "3": 1, "4": 1, "5": 1, "6": 1,       # low cards: +1
    "7": 0, "8": 0, "9": 0,                       # neutral cards: 0
    "10": -1, "J": -1, "Q": -1, "K": -1, "A": -1, # high cards: -1
}

def hilo_value(rank):
    """Return +1, 0 or -1 for a valid, normalised rank."""
    return HILO_VALUES[rank]          # rank must already be checked by normalise_rank in part 3.

5. Everything that must change together changes inside one function. Invalid cards change nothing.

In [5]:
def count_card(state, label):
    """Update the shoe for one card. Returns True if counted, False if skipped."""
    rank = normalise_rank(label)          # clean up the label first
    if rank is None:                      # unreadable or invalid card
        return False                      # skip it: count and cards_seen stay in sync
    state["running_count"] += hilo_value(rank)   # add +1, 0 or -1 to the count
    state["cards_seen"] += 1                      # then record that we saw a card
    return True                           # tell the caller it was counted

6. Decks left

In [6]:
def decks_left(state, minimum=0.25):
    """Estimate decks remaining, never lower than `minimum`."""
    used = state["cards_seen"] / 52                    # cards seen -> part of decks used
    remaining = state["decks_total"] - used            # decks still in the shoe
    return max(remaining, minimum)                     # floor avoids divide-by-zero

7. Validate true count and rounding for strategy decisions

In [7]:
import math                                    # needed for math.floor

def true_count(state):
    """True count = running count divided by decks left."""
    return state["running_count"] / decks_left(state)  # same units: count per deck

def true_count_floored(state):
    """Whole-number true count (rounded down) for strategy lookups."""
    return math.floor(true_count(state))       # rounding for true count ie -2.9 -> -3 and 1.7 -> 1

8. Same image counted twice when re-running notebook cell

In [8]:
def count_image(state, image_id, label):
    """Count a card from one image, but only the first time that image is seen."""
    if image_id in state["processed_ids"]:    # already counted this image?
        return False                          # yes: ignore it so we don't double count
    counted = count_card(state, label)        # try to count it
    if counted:                               # only remember images that were valid
        state["processed_ids"].add(image_id)  # note it so re-runs skip it
    return counted                            # report what happened

9. Processing several images at once

In [9]:
def count_batch(state, results):
    """results is a list of (image_id, label) pairs from your classifier."""
    skipped = []                              # collect problems to show the user
    for image_id, label in results:           # go through each classified image
        if not count_image(state, image_id, label):   # False = duplicate or invalid
            skipped.append((image_id, label)) # remember which one was skipped
    return skipped                            # caller can print or inspect these

10. Showing results

In [10]:
def show(state):
    """Print the current count information."""
    print(f"Running count : {state['running_count']:+d}")          # +d forces a sign: +3 / -2
    print(f"True count    : {true_count(state):+.2f}")             # 2 decimal places
    print(f"Decks left    : {decks_left(state):.2f} / {state['decks_total']}")
    print(f"Cards seen    : {state['cards_seen']}")                # sanity check number
    print(f"True count (whole): {true_count_floored(state):+d}")   # integer used for strategy/bets

11. Starting a new shoe after a shuffle

In [11]:
def reset(state):
    """Wipe the count but keep the same number of decks."""
    fresh = new_shoe(state["decks_total"])    # make a clean set of values
    state.clear()                             # empty the existing dictionary...
    state.update(fresh)                       # ...and refill it, so other code sees the change

In [12]:
state = new_shoe(6)    # fixed 6 decks so it runs without asking; use ask_decks() later

In [13]:
reset(state)                                   # start from an empty count (keeps the same number of decks)

ranks = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]   # the 13 ranks in one suit
results = []                                   # will hold 52 (image name, card) pairs
for suit in range(4):                          # 4 suits...
    for rank in ranks:                         # ...of 13 ranks each = 52 cards
        results.append((f"img{len(results) + 1}.jpg", rank))   # unique name: img1.jpg ... img52.jpg

skipped = count_batch(state, results)          # count the whole deck
show(state)                                    # display the results
print("Skipped:", skipped)                     # should be empty: nothing bad in this list

assert state["running_count"] == 0, "a full deck must sum to 0"      # Hi-Lo is balanced, so a full deck is 0
assert state["cards_seen"] == 52, "all 52 cards should be counted"   # nothing should have been skipped
assert skipped == [], "no card should have been skipped"
print("52-card test PASSED")

Running count : +0
True count    : +0.00
Decks left    : 5.00 / 6
Cards seen    : 52
True count (whole): +0
Skipped: []
52-card test PASSED
